# 1. Import Libraries

This section imports the Python libraries required for data manipulation, visualization, and machine learning throughout the project.

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



# 2. Load the Olist Datasets

The Olist dataset consists of multiple related tables. For this project, we use the four core datasets required to build a customer churn prediction model:

- Customers
- Orders
- Order Items
- Payments

In [6]:
customers = pd.read_csv(r"C:\Users\LENOVO\Documents\Data Science Projects\churn prediction\Data\Raw\olist_customers_dataset.csv")
orders = pd.read_csv(r"C:\Users\LENOVO\Documents\Data Science Projects\churn prediction\Data\Raw\olist_orders_dataset.csv")
order_items = pd.read_csv(r"C:\Users\LENOVO\Documents\Data Science Projects\churn prediction\Data\Raw\olist_order_items_dataset.csv")
payments = pd.read_csv(r"C:\Users\LENOVO\Documents\Data Science Projects\churn prediction\Data\Raw\olist_order_payments_dataset.csv")

print(customers.shape, orders.shape, order_items.shape, payments.shape)

(99441, 5) (99441, 8) (112650, 7) (103886, 5)


# 3. Data Inspection

Before preprocessing, we inspect each dataset to understand:

- Number of rows and columns
- Data types
- Missing values
- Overall structure

This helps identify any cleaning or transformation needed before feature engineering.

In [7]:
customers.info()
orders.info()
order_items.info()
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4

In [8]:
customers.isnull().sum()
orders.isnull().sum()
order_items.isnull().sum()
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

# 4. Convert Date Columns

The order timestamps are initially stored as text. Converting them to datetime format enables time-based calculations such as customer recency, tenure, and churn labeling.

In [9]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [10]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

# 5. Filter Delivered Orders

Only completed (delivered) orders are retained because they represent actual customer purchases. Cancelled or incomplete orders are excluded to ensure accurate customer behavior analysis.

In [11]:
orders = orders[orders['order_status'] == 'delivered'].copy()

print("Delivered Orders:", orders.shape)

Delivered Orders: (96478, 8)


# 6. Merge Customer and Order Data

The customer and order datasets are merged using `customer_id`. This combines customer information with purchase history while preserving `customer_unique_id`, which uniquely identifies each customer across multiple orders.

In [12]:
customer_orders = customers.merge(
    orders,
    on="customer_id",
    how="inner"
)

print(customer_orders.shape)
customer_orders.head()

(96478, 12)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15


# 7. Create Order-Level Features

Each order may contain multiple products and payment records. The data is aggregated to calculate:

- Total order value
- Total freight cost
- Number of items purchased
- Total payment amount
- Payment installments

In [13]:
order_value = (
    order_items
    .groupby('order_id')
    .agg(
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        total_items=('order_item_id', 'count')
    )
    .reset_index()
)

order_value.head()

,order_id,total_price,total_freight,total_items
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1


In [14]:
payment_summary = (
    payments
    .groupby('order_id')
    .agg(
        payment_value=('payment_value', 'sum'),
        payment_installments=('payment_installments', 'max')
    )
    .reset_index()
)

payment_summary.head()

,order_id,payment_value,payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,3
2,000229ec398224ef6ca0657da4fc703e,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3


# 8. Merge Order-Level Features into Customer Data

The aggregated order-level features are merged back into the `customer_orders` dataset using `order_id`. This enriches each customer order with additional information such as:

- Total order value
- Total freight cost
- Number of items purchased
- Total payment amount
- Payment installments

By combining these features with customer and order information, we create a comprehensive dataset that serves as the foundation for customer-level feature engineering and churn prediction.
)

In [15]:
customer_orders = customer_orders.merge(
    order_value,
    on='order_id',
    how='left'
)

customer_orders = customer_orders.merge(
    payment_summary,
    on='order_id',
    how='left'
)
customer_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   customer_id                    96478 non-null  object        
 1   customer_unique_id             96478 non-null  object        
 2   customer_zip_code_prefix       96478 non-null  int64         
 3   customer_city                  96478 non-null  object        
 4   customer_state                 96478 non-null  object        
 5   order_id                       96478 non-null  object        
 6   order_status                   96478 non-null  object        
 7   order_purchase_timestamp       96478 non-null  datetime64[ns]
 8   order_approved_at              96464 non-null  datetime64[ns]
 9   order_delivered_carrier_date   96476 non-null  datetime64[ns]
 10  order_delivered_customer_date  96470 non-null  datetime64[ns]
 11  order_estimated

### Observation

The merged dataset now contains customer details, order information, and aggregated order-level features in a single table. This integrated dataset enables the calculation of customer-level metrics such as total spending, purchase frequency, recency, and tenure, which are essential for building the churn prediction model.

## 9.1 Handle Missing Values

Before aggregating the data at the customer level, the missing values in the payment-related columns are handled.

Only one order has missing payment information. Since the number of missing records is negligible, the missing values are replaced with **0**, ensuring that the aggregation process is not affected.

In [16]:
customer_orders['payment_value'] = customer_orders['payment_value'].fillna(0)
customer_orders['payment_installments'] = customer_orders['payment_installments'].fillna(0)

customer_orders[['payment_value', 'payment_installments']].isnull().sum()

payment_value           0
payment_installments    0
dtype: int64

# 9.2 Customer-Level Feature Engineering

The dataset currently contains one row per completed order. However, churn prediction requires one record per customer.

The order-level data is aggregated using `customer_unique_id` to generate customer-level features that summarize purchasing behavior.

The engineered features include:

- Total number of orders
- Total spending
- Average order value
- Average number of items purchased
- Total freight cost
- Average payment installments
- First purchase date
- Last purchase date

These features will serve as the foundation for customer behavior analysis and machine learning.

In [17]:
customer_features = (
    customer_orders
    .groupby('customer_unique_id')
    .agg(
        total_orders=('order_id', 'count'),
        total_spent=('payment_value', 'sum'),
        avg_order_value=('payment_value', 'mean'),
        avg_items=('total_items', 'mean'),
        total_freight=('total_freight', 'sum'),
        avg_installments=('payment_installments', 'mean'),
        first_purchase=('order_purchase_timestamp', 'min'),
        last_purchase=('order_purchase_timestamp', 'max'),
        customer_state=('customer_state', 'first'),
        customer_city=('customer_city', 'first')
    )
    .reset_index()
)

customer_features.head()

,customer_unique_id,total_orders,total_spent,avg_order_value,avg_items,total_freight,avg_installments,first_purchase,last_purchase,customer_state,customer_city
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,1.0,12.00,8.0,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,cajamar
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,1.0,8.29,1.0,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,osasco
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,1.0,17.22,8.0,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,sao jose
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,1.0,17.63,4.0,2017-10-12 20:29:41,2017-10-12 20:29:41,PA,belem
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,1.0,16.89,6.0,2017-11-14 19:45:42,2017-11-14 19:45:42,SP,sorocaba


In [18]:
# Snapshot cutoff: 90 days before the last order date in the whole dataset
snapshot_date = orders['order_purchase_timestamp'].max() - pd.Timedelta(days=90)
print("Snapshot date:", snapshot_date)

# Keep only customers whose first purchase happened early enough 
# to have had a fair 90-day chance to return before the data collection ended
customer_features = customer_features[customer_features['first_purchase'] <= snapshot_date].copy()

print("Customers remaining after filtering:", customer_features.shape[0])

Snapshot date: 2018-05-31 15:00:37
Customers remaining after filtering: 75320


# 10. Create RFM (Recency, Frequency, Monetary) Features

RFM (Recency, Frequency, Monetary) is a widely used customer analytics technique that summarizes purchasing behavior.

The three metrics are defined as follows:

- **Recency (R):** Number of days since the customer's most recent purchase.
- **Frequency (F):** Total number of completed purchases made by the customer.
- **Monetary (M):** Total amount spent by the customer.

These features are among the most influential predictors of customer churn and provide valuable insights into customer engagement and loyalty.

In [19]:
# Recency: measured against the snapshot date (not the dataset's last date),
# to avoid unfairly labeling recent customers as churned
customer_features['recency'] = (
    snapshot_date - customer_features['last_purchase']
).dt.days

# Frequency
customer_features['frequency'] = customer_features['total_orders']

# Monetary
customer_features['monetary'] = customer_features['total_spent']

customer_features[['customer_unique_id','recency','frequency','monetary']].head()

,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,21,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,24,1,27.19
2,0000f46a3911fa3c0805444483337064,446,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,230,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,197,1,196.89


### Observation

The RFM metrics have been successfully created. These features summarize customer purchasing behavior and are widely used in customer segmentation and churn prediction. Customers with high recency values and low purchase frequency are generally more likely to churn.

# 11. Calculate Customer Tenure

Customer tenure represents the duration of the relationship between a customer and the business.

It is calculated as the number of days between the customer's first and most recent completed purchase.

Longer tenure often indicates stronger customer loyalty and may influence churn behavior.

In [20]:
customer_features['tenure'] = (
    customer_features['last_purchase']
    - customer_features['first_purchase']
).dt.days

customer_features[['customer_unique_id','tenure']].head()

,customer_unique_id,tenure
0,0000366f3b9a7992bf8c76cfdf3221e2,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,0
2,0000f46a3911fa3c0805444483337064,0
3,0000f6ccb0745a6a4b88665a16c9f078,0
4,0004aac84e0df4da2b147fca70cf8255,0


### Observation

Customer tenure has been successfully calculated. This feature measures how long a customer has been active and provides additional information about customer loyalty and purchasing history.

# 12. Generate the Churn Label

The Olist dataset does not contain a predefined churn label. Therefore, a custom churn definition is created based on customer inactivity.

For this project, a customer is classified as **churned** if they have not made another purchase within **90 days** of their most recent purchase.

This creates the binary target variable required for supervised machine learning.

In [21]:
customer_features['churn'] = (
    customer_features['recency'] > 90
).astype(int)

customer_features[['customer_unique_id','recency','churn']].head()

,customer_unique_id,recency,churn
0,0000366f3b9a7992bf8c76cfdf3221e2,21,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,24,0
2,0000f46a3911fa3c0805444483337064,446,1
3,0000f6ccb0745a6a4b88665a16c9f078,230,1
4,0004aac84e0df4da2b147fca70cf8255,197,1


### Observation

A binary target variable has been created:

- **0** → Active Customer
- **1** → Churned Customer

This target variable will be used to train and evaluate the churn prediction models.

# 13. Verify the Final Customer Dataset

Before performing exploratory data analysis and building machine learning models, the final customer-level dataset is verified.

The following checks are performed:

- Dataset structure
- Summary statistics
- Sample records
- Churn class distribution

These checks ensure that the engineered features and target variable have been created correctly.

In [22]:
customer_features.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75320 entries, 0 to 93357
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_unique_id  75320 non-null  object        
 1   total_orders        75320 non-null  int64         
 2   total_spent         75320 non-null  float64       
 3   avg_order_value     75320 non-null  float64       
 4   avg_items           75320 non-null  float64       
 5   total_freight       75320 non-null  float64       
 6   avg_installments    75320 non-null  float64       
 7   first_purchase      75320 non-null  datetime64[ns]
 8   last_purchase       75320 non-null  datetime64[ns]
 9   customer_state      75320 non-null  object        
 10  customer_city       75320 non-null  object        
 11  recency             75320 non-null  int64         
 12  frequency           75320 non-null  int64         
 13  monetary            75320 non-null  float64       


In [23]:
customer_features.describe(include='all')

,customer_unique_id,total_orders,total_spent,avg_order_value,avg_items,total_freight,avg_installments,first_purchase,last_purchase,customer_state,customer_city,recency,frequency,monetary,tenure,churn
count,75320,75320.000000,75320.000000,75320.000000,75320.000000,75320.000000,75320.000000,75320,75320,75320,75320,75320.000000,75320.000000,75320.000000,75320.000000,75320.000000
unique,75320,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27,3877,NaN,NaN,NaN,NaN,NaN
top,0000366f3b9a7992bf8c76cfdf3221e2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SP,sao paulo,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30794,11256,NaN,NaN,NaN,NaN,NaN
mean,NaN,1.038768,165.254368,159.615235,1.141642,23.167965,2.943107,2017-11-15 15:38:02.783736064,2017-11-18 21:25:59.708828928,NaN,NaN,193.206585,1.038768,165.254368,3.228691,0.730297
min,NaN,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,2016-09-15 12:16:38,2016-09-15 12:16:38,NaN,NaN,-90.000000,1.000000,0.000000,0.000000,0.000000
25%,NaN,1.000000,63.130000,62.340000,1.000000,14.100000,1.000000,2017-08-10 00:16:28.249999872,2017-08-14 13:45:01.249999872,NaN,NaN,84.000000,1.000000,63.130000,0.000000,0.000000
50%,NaN,1.000000,107.780000,105.280000,1.000000,17.060000,2.000000,2017-12-03 19:18:34.500000,2017-12-06 01:17:32.500000,NaN,NaN,176.000000,1.000000,107.780000,0.000000,1.000000
75%,NaN,1.000000,182.940000,175.952500,1.000000,25.380000,4.000000,2018-03-05 10:06:31.249999872,2018-03-07 19:19:35,NaN,NaN,290.000000,1.000000,182.940000,0.000000,1.000000
max,NaN,15.000000,13664.080000,13664.080000,21.000000,1002.290000,24.000000,2018-05-31 14:38:55,2018-08-28 21:56:12,NaN,NaN,623.000000,15.000000,13664.080000,633.000000,1.000000


In [24]:
customer_features.head()

,customer_unique_id,total_orders,total_spent,avg_order_value,avg_items,total_freight,avg_installments,first_purchase,last_purchase,customer_state,customer_city,recency,frequency,monetary,tenure,churn
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,1.0,12.00,8.0,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,cajamar,21,1,141.90,0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,1.0,8.29,1.0,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,osasco,24,1,27.19,0,0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,1.0,17.22,8.0,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,sao jose,446,1,86.22,0,1
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,1.0,17.63,4.0,2017-10-12 20:29:41,2017-10-12 20:29:41,PA,belem,230,1,43.62,0,1
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,1.0,16.89,6.0,2017-11-14 19:45:42,2017-11-14 19:45:42,SP,sorocaba,197,1,196.89,0,1


In [25]:
customer_features['churn'].value_counts()

churn
1    55006
0    20314
Name: count, dtype: int64

In [26]:
customer_features['churn'].value_counts(normalize=True) * 100

churn
1    73.02974
0    26.97026
Name: proportion, dtype: float64

### Observation

Initial churn labeling using the dataset's maximum date as the reference point 
produced a churn rate of ~80%, which was misleadingly high. This occurred because 
customers whose last purchase fell near the end of the data collection window 
(mid-2018) had insufficient time remaining in the dataset to make a repeat purchase, 
causing them to be incorrectly labeled as churned — a form of right-censoring bias.

To correct this, a snapshot date was defined as 90 days before the dataset's last 
recorded order (2018-05-31). Only customers whose first purchase occurred on or 
before this snapshot date were retained (75,320 of ~93,000 customers), ensuring 
every included customer had a fair 90-day window to make a repeat purchase.

After this correction, the churn rate is **73.0%** (55,006 churned vs. 20,314 active). 
This remains meaningfully imbalanced, but is consistent with Olist's known business 
reality — the platform has a very low customer repeat-purchase rate, so a high 
proportion of true "one-and-done" buyers is expected rather than an artifact of 
labeling bias. Given this, accuracy alone would not be an appropriate evaluation 
metric, since a model predicting "churned" for every customer would already score 
~73%. Precision, Recall, F1-score, and ROC-AUC will be used instead, and class 
weighting or SMOTE will be considered during model training.

In [27]:
customer_features.to_csv(r"C:\Users\LENOVO\Documents\Data Science Projects\churn prediction\Data\Processed\customer_features.csv", index=False)
